## Import Lib

In [1]:
import pandas as pd
from datetime import datetime, timedelta
import os
import numpy as np
import zipfile

In [2]:
path = r"C:\Users\Akhil\Downloads\project\TradeX_raw\STcon"


os.chdir(path)
print("Current Working Directory:", os.getcwd())


xlsx_files = [f for f in os.listdir(path) if f.endswith('.xlsx')]

if len(xlsx_files) != 3:
    raise ValueError("There should be exactly 3 .xlsx files in the directory.")

Current Working Directory: C:\Users\Akhil\Downloads\project\TradeX_raw\STcon


In [3]:
# Read the files
st_1 = pd.read_excel(xlsx_files[0])
st_2 = pd.read_excel(xlsx_files[1])
st_3 = pd.read_excel(xlsx_files[2])

# Concatenate
concat_df = pd.concat([st_1, st_2, st_3], ignore_index=True)

print("Length of st_1:", len(st_1))
print("Length of st_2:", len(st_2))
print("Length of st_3:", len(st_3))
print("Length of concatenated DataFrame:", len(concat_df))

# Filter out unwanted rows
stringee = concat_df[concat_df['End of Call code'] != 'CAN_NOT_MAKE_CALL']

# Generate dynamic filename S_DD_MM.csv
yesterday = datetime.today()-timedelta(days=1)
file_name = f"S_{yesterday.day:02d}_{yesterday.month:02d}.csv"

# Build full output path
output_path = os.path.join(
    r"C:\Users\Akhil\Downloads\project\TradeX_raw",
    file_name
)

# Save CSV
stringee.to_csv(output_path, index=False)
print(f"CSV saved to: {output_path}")

Length of st_1: 702
Length of st_2: 169
Length of st_3: 798
Length of concatenated DataFrame: 1669
CSV saved to: C:\Users\Akhil\Downloads\project\TradeX_raw\S_30_03.csv


c:\Users\Akhil\Office\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Akhil\Office\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\Akhil\Office\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [4]:
st_3.dtypes

ID                          object
Customer number             object
Hotline                      int64
Call type                   object
Start time          datetime64[ns]
End time            datetime64[ns]
Queue duration              object
Answer duration             object
Account                     object
Hold duration               object
Contact                     object
Company ID                 float64
End of Call code            object
Call status                 object
dtype: object

In [5]:

path = r"C:\Users\Akhil\Downloads\project\TradeX_raw\TTcon"
os.chdir(path)

print("Current Working Directory:", os.getcwd())

# Find the single ZIP file
zip_files = [f for f in os.listdir(path) if f.endswith('.zip')]
if len(zip_files) != 1:
    raise ValueError("There should be exactly 1 .zip file in the directory.")

zip_path = os.path.join(path, zip_files[0])

# Open ZIP and read the only CSV inside it
with zipfile.ZipFile(zip_path, 'r') as z:
    inner_files = z.namelist()

    csv_inside = [f for f in inner_files if f.endswith('.csv')]
    if len(csv_inside) != 1:
        raise ValueError("There should be exactly 1 CSV inside the ZIP.")

    df = pd.read_csv(z.open(csv_inside[0]))

df.head()


Current Working Directory: C:\Users\Akhil\Downloads\project\TradeX_raw\TTcon


,Direction,Call Status,Call ID,Time,Customer Number,Customer Name,Call Solution,DID,Agent Number,Answered By Agent,...,AMD Status,Inbound Duration,Outbound Duration,Call Sub Type,Wait Duration,Answer Duration,Skill ID,Skill Name,DTMF,Dial Assist Score (%)
0,Inbound,Missed,"""1774885792.127365""",2026-03-30 21:19:52,919653040864,NaN,Incoming,917969532205,NaN,NaN,...,NaN,00:00:06,00:00:00,NaN,00:00:00,00:00:00,NaN,NaN,NaN,NaN
1,Inbound,Missed,"""1774885309.127342""",2026-03-30 21:11:49,918585916957,NaN,Incoming,917969532200,NaN,NaN,...,NaN,00:00:06,00:00:00,NaN,00:00:00,00:00:00,NaN,NaN,NaN,NaN
2,Outbound,Missed,"""1774883787.1929952""",2026-03-30 20:46:27,919691191881,NaN,Clicktocall,917969532221,+919568458745,shiv-Extension,...,NaN,00:00:16,00:00:00,NaN,00:00:00,00:00:00,NaN,NaN,NaN,NaN
3,Outbound,Missed,"""1774882678.123885""",2026-03-30 20:27:58,919334658987,NaN,Clicktocall,917969532244,+916958419685,251 Chandan-Extension,...,NaN,00:00:05,00:00:00,NaN,00:00:00,00:00:00,NaN,NaN,NaN,NaN
4,Outbound,Missed,"""1774882628.120089""",2026-03-30 20:27:08,919125308105,NaN,Clicktocall,917969532244,+916958419685,251 Chandan-Extension,...,NaN,00:00:06,00:00:00,NaN,00:00:00,00:00:00,NaN,NaN,NaN,NaN


In [6]:
T2 = df.copy()
T2 = T2.dropna(subset=["Call Flow"])
T2 = T2.rename(columns={"Client Number": "Customer Number", "Status": "Call Status"})

# Connected agent
T2["Connected to Agent"] = T2["Call Flow"].str.extract(r'Agent:\s*([^(\-]+)')

# Correct date extraction (YYYY-MM-DD)
T2["Call Start Date"] = T2["Call Flow"].str.extract(r'\((\d{4}-\d{2}-\d{2})\s')

# Correct time extraction
T2["Call Start Time"] = T2["Call Flow"].str.extract(r'\d{4}-\d{2}-\d{2}\s(\d{2}:\d{2}:\d{2})')

# Remove rows where date missing
T2 = T2.dropna(subset=["Call Start Date"])

# Correct customer number extraction
# T2["Customer Number"] = T2["Call Flow"].str.extract(r'Customer:\s*([+0-9X]+)')

# Duration fields
T2["Answer Duration (HH:MM:SS)"] = (
    pd.to_timedelta(T2["Outbound Duration"], errors="coerce")
    .fillna(pd.to_timedelta(0))
    .astype(str)
    .str[-8:]
)


T2["Total Call Duration (HH:MM:SS)"] = ( pd.to_timedelta(T2["Call Duration"]) .astype(str) .str[-8:] .replace("NaT", "00:00:00") )

T2["Hold Duration (HH:MM:SS)"] = "00:00:00"

# Convert date format
T2["Call Start Date"] = pd.to_datetime(T2["Call Start Date"]).dt.strftime('%Y-%m-%d')

# Final columns
T2 = T2[['Call Start Date', 'Connected to Agent', 'Customer Number', 'Call Status',
         'Answer Duration (HH:MM:SS)', 'Hold Duration (HH:MM:SS)',
         'Total Call Duration (HH:MM:SS)', 'Call Start Time']]


In [7]:
path = r"C:\Users\Akhil\Downloads\project\TradeX_raw"

#\pyton automation\Tradex_dialer_raw
# Change the working directory
os.chdir(path)

# Confirm the new working directory
print("Current Working Directory:", os.getcwd())

Current Working Directory: C:\Users\Akhil\Downloads\project\TradeX_raw


## Adjustment

In [8]:
# voiso = pd.read_csv('VT_12-16.csv',low_memory=False)
tata = T2.copy()
know = pd.read_csv('K_30_03.csv',low_memory=False)
Qconn = pd.read_csv('Q_18_07.csv',low_memory=False)
stringee = stringee.copy()

## Check_point_1

In [9]:
# print("voiso:", voiso['Date and time'].unique()[:5])
print("tata:", tata['Call Start Date'].unique()[:5])
print("know:", know['Date and Time'].unique()[:5])
print("Qconn:", Qconn['Date time'].unique()[:5])
print("stringee:", stringee['Start time'].unique()[:5])

tata: ['2026-03-30']
know: ['2026-03-30 21:50:48' '2026-03-30 21:05:23' '2026-03-30 20:32:48'
 '2026-03-30 20:17:31' '2026-03-30 20:16:10']
Qconn: []
stringee: <DatetimeArray>
['2026-03-30 18:00:07.454000', '2026-03-30 17:57:23.628000',
 '2026-03-30 17:23:04.845000', '2026-03-30 16:59:18.215000',
 '2026-03-30 16:58:17.891000']
Length: 5, dtype: datetime64[ns]


In [10]:
stringee

,ID,Customer number,Hotline,Call type,Start time,End time,Queue duration,Answer duration,Account,Hold duration,Contact,Company ID,End of Call code,Call status
0,call-vn-1-9EYTIVYVSP-1772159776507,917982995504,917949152410,Inbound calls,2026-03-30 18:00:07.454,2026-03-30 18:00:07.465,00:00:00,00:00:00,NaN,00:00:00,Contact 917982995504,NaN,USER_END_CALL,Stopped at IVR
1,call-vn-1-9EYTIVYVSP-1772159775687,917982995504,917949152410,Inbound calls,2026-03-30 17:57:23.628,2026-03-30 17:57:23.641,00:00:00,00:00:00,NaN,00:00:00,Contact 917982995504,NaN,USER_END_CALL,Stopped at IVR
2,call-vn-1-9EYTIVYVSP-1772159764348,918878363662,917949152411,Inbound calls,2026-03-30 17:23:04.845,2026-03-30 17:23:04.855,00:00:00,00:00:00,NaN,00:00:00,NaN,NaN,USER_END_CALL,Stopped at IVR
3,call-vn-1-9EYTIVYVSP-1772159754557,917023333365,917949152410,Outbound calls,2026-03-30 16:59:18.215,2026-03-30 16:59:33.720,00:00:15,00:00:00,myron.f@nexora.live,00:00:00,Contact 917023333365,NaN,480 Temporarily Unavailable,Missed
4,call-vn-1-9EYTIVYVSP-1772159754084,918439171586,917949152410,Outbound calls,2026-03-30 16:58:17.891,2026-03-30 16:58:31.913,00:00:14,00:00:00,myron.f@nexora.live,00:00:00,Contact 918439171586,NaN,USER_END_CALL,Missed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1664,call-vn-1-9EYTIVYVSP-1772159281030,917559337579,917949152408,Outbound calls,2026-03-30 08:12:31.663,2026-03-30 08:13:15.938,00:00:44,00:00:00,hritika@nexora.live,00:00:00,NaN,NaN,404 Not Found,Missed
1665,call-vn-1-9EYTIVYVSP-1772159279770,919131943813,917949152408,Outbound calls,2026-03-30 08:11:30.367,2026-03-30 08:11:58.053,00:00:27,00:00:00,hritika@nexora.live,00:00:00,NaN,NaN,480 Temporarily Unavailable,Missed
1666,call-vn-1-9EYTIVYVSP-1772159278762,919999495915,917949152408,Outbound calls,2026-03-30 08:10:40.261,2026-03-30 08:12:00.904,00:00:30,00:00:49,kawaljit.k@nexora.live,00:00:00,NaN,NaN,USER_END_CALL,Answered
1667,call-vn-1-9EYTIVYVSP-1772159278332,919140107280,917949152408,Outbound calls,2026-03-30 08:10:18.778,2026-03-30 08:10:44.899,00:00:26,00:00:00,hritika@nexora.live,00:00:00,NaN,NaN,480 Temporarily Unavailable,Missed


In [11]:
stringee['Start time'] = pd.to_datetime(stringee['Start time']) + pd.Timedelta(hours=1, minutes=30)

In [12]:
stringee.columns

Index(['ID', 'Customer number', 'Hotline', 'Call type', 'Start time',
       'End time', 'Queue duration', 'Answer duration', 'Account',
       'Hold duration', 'Contact', 'Company ID', 'End of Call code',
       'Call status'],
      dtype='object')

In [13]:
stringee['Answer duration'].dtype

dtype('O')

In [14]:
# Function to convert durations in hh:mm:ss format to timedelta
def duration_to_timedelta(duration):
    hours, minutes, seconds = map(int, duration.split(':'))
    return timedelta(hours=hours, minutes=minutes, seconds=seconds)

# Convert the 'Queue duration' and 'Answer duration' columns to timedelta
stringee['Queue Duration (timedelta)'] = stringee['Queue duration'].apply(duration_to_timedelta)
stringee['Answer Duration (timedelta)'] = stringee['Answer duration'].apply(duration_to_timedelta)

# Calculate the total duration as timedelta
stringee['Total Duration (timedelta)'] = stringee['Queue Duration (timedelta)'] + stringee['Answer Duration (timedelta)']

# Convert the total duration back to hh:mm:ss format (remove "0 days")
stringee['Total Duration'] = stringee['Total Duration (timedelta)'].apply(lambda x: str(x).split(", ")[-1])

# Drop intermediate timedelta columns if not needed
stringee = stringee.drop(columns=['Queue Duration (timedelta)', 'Answer Duration (timedelta)', 'Total Duration (timedelta)'])


## ETL

In [15]:
new_tata = tata[['Call Start Date', 'Connected to Agent','Call Status','Answer Duration (HH:MM:SS)',
                 'Hold Duration (HH:MM:SS)','Total Call Duration (HH:MM:SS)','Call Start Time','Customer Number']]

In [16]:

new_Know = know[['Date and Time', 'Agent Name','Call Status', 'Talk Time (hh:mm:ss)', 'Hold Time (hh:mm:ss)','Total Call Duration (hh:mm:ss)','Customer']]

In [17]:
# new_voiso =  voiso[['Date and time','Agent(s)','Disposition','Talk time','Duration','DNIS/To']]

In [18]:
new_qconn = Qconn[['Date time','Agent Mobile','Call Event','Transfer Duration','Duration','User Mobile']]

In [19]:
new_string = stringee[['Start time','Account','Call status','Answer duration','Hold duration','Total Duration','Customer number']]

## Voiso date 

In [20]:
# new_voiso['Date and time'].unique()

In [21]:
# # Function to convert 'Date and time' from mm/dd/yyyy HH:mm:ss to yyyy/mm/dd HH:mm:ss
# def fix_voiso_datetime(df, col_name):
#     # Convert using pd.to_datetime, specify the format as mm/dd/yyyy
#     df[col_name] = pd.to_datetime(df[col_name], errors='coerce', format='%m/%d/%Y %H:%M:%S') #
    
#     # Format the datetime to 'yyyy/mm/dd HH:mm:ss'
#     df[col_name] = df[col_name].dt.strftime('%Y/%m/%d %H:%M:%S')
    
#     return df

# # Apply the function to 'Date and time' column in Voiso
# new_voiso = fix_voiso_datetime(new_voiso, 'Date and time')

# # Verify the result
# print("Voiso Date and time after fixing format:", new_voiso['Date and time'].unique())


## string date format

In [22]:
#####check
new_string['Start time'].head()

0   2026-03-30 19:30:07.454
1   2026-03-30 19:27:23.628
2   2026-03-30 18:53:04.845
3   2026-03-30 18:29:18.215
4   2026-03-30 18:28:17.891
Name: Start time, dtype: datetime64[ns]

In [23]:
# Function to fix 'new_string' Date Time format
def fix_string_datetime(date_str):
    try:
        # Parse the date and time, assuming format mm/dd/yyyy hh:mm:ss AM/PM
        parsed_date = pd.to_datetime(date_str, errors='coerce', format='%Y-%m-%d %H:%M:%S.%f')
        return parsed_date
    except Exception as e:
        return pd.NaT  # Return NaT for any unparseable date

# Apply this function to the 'Date and Time' column in 'new_string'
new_string['Start time'] = new_string['Start time'].apply(fix_string_datetime)

# Now format it to yyyy/mm/dd HH:mm:ss
new_string['Start time'] = new_string['Start time'].dt.strftime('%Y/%m/%d %H:%M:%S')

# Check the result
print("String unique date formats after fixing:", new_string['Start time'].head())

String unique date formats after fixing: 0    2026/03/30 19:30:07
1    2026/03/30 19:27:23
2    2026/03/30 18:53:04
3    2026/03/30 18:29:18
4    2026/03/30 18:28:17
Name: Start time, dtype: object


C:\Users\Akhil\AppData\Local\Temp\ipykernel_13804\1218228788.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_string['Start time'] = new_string['Start time'].apply(fix_string_datetime)
C:\Users\Akhil\AppData\Local\Temp\ipykernel_13804\1218228788.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_string['Start time'] = new_string['Start time'].dt.strftime('%Y/%m/%d %H:%M:%S')


In [24]:
new_Know['Date and Time'].head()


0    2026-03-30 21:50:48
1    2026-03-30 21:05:23
2    2026-03-30 20:32:48
3    2026-03-30 20:17:31
4    2026-03-30 20:16:10
Name: Date and Time, dtype: object

In [25]:
new_qconn['Date time'].head()

Series([], Name: Date time, dtype: object)

In [26]:
# Convert 'Date and Time' column in new_Know to datetime format
new_qconn['Date time'] = pd.to_datetime(new_qconn['Date time'], errors='coerce', format='%Y-%m-%d %H:%M:%S')
new_Know['Date and Time'] = pd.to_datetime(new_Know['Date and Time'], errors='coerce', format='%Y-%m-%d %H:%M:%S') #%d/%m/%Y

# Now check the type of 'Date' again in both dataframes
print("Data type of 'Date' in new_qconn after conversion:", new_qconn['Date time'].dtype)
print("Data type of 'Date' in new_Know after conversion:", new_Know['Date and Time'].dtype)

# Extract date and call start time for new_Know
new_Know['Date'] = new_Know['Date and Time'].dt.date  # Extract date part
new_Know['Call Start Time'] = new_Know['Date and Time'].dt.strftime('%H:%M:%S')  # Extract time part

# Extract date and call start time for new_qconn (if you haven't done this yet)
new_qconn['Date'] = new_qconn['Date time'].dt.date  # Extract date part
new_qconn['Call Start Time'] = new_qconn['Date time'].dt.strftime('%H:%M:%S')  # Extract time part

# Check the resulting dataframes to ensure extraction is successful
print(new_qconn[['Date', 'Call Start Time']].head())
print(new_Know[['Date', 'Call Start Time']].head())


Data type of 'Date' in new_qconn after conversion: datetime64[ns]
Data type of 'Date' in new_Know after conversion: datetime64[ns]
Empty DataFrame
Columns: [Date, Call Start Time]
Index: []
         Date Call Start Time
0  2026-03-30        21:50:48
1  2026-03-30        21:05:23
2  2026-03-30        20:32:48
3  2026-03-30        20:17:31
4  2026-03-30        20:16:10


C:\Users\Akhil\AppData\Local\Temp\ipykernel_13804\4068174409.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_Know['Date and Time'] = pd.to_datetime(new_Know['Date and Time'], errors='coerce', format='%Y-%m-%d %H:%M:%S') #%d/%m/%Y
C:\Users\Akhil\AppData\Local\Temp\ipykernel_13804\4068174409.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_Know['Date'] = new_Know['Date and Time'].dt.date  # Extract date part
C:\Users\Akhil\AppData\Local\Temp\ipykernel_13804\4068174409.py:11: SettingWithCopyWar

In [27]:
# Find rows where 'Date time' is NaT
invalid_dates = new_qconn[new_qconn['Date time'].isna()]
print(invalid_dates)


Empty DataFrame
Columns: [Date time, Agent Mobile, Call Event, Transfer Duration, Duration, User Mobile, Date, Call Start Time]
Index: []


In [28]:
(new_qconn['Date'].unique())

array([], dtype=object)

In [29]:
# Check the data type of the 'Date' column
print("Data type of 'Date' in new_qconn:", new_qconn['Date time'].dtype)
print("Data type of 'Date' in new_Know:", new_Know['Date and Time'].dtype)


Data type of 'Date' in new_qconn: datetime64[ns]
Data type of 'Date' in new_Know: datetime64[ns]


In [30]:
# Function to separate Date and Call Start Time
def split_date_time(df, col_name):
    # Convert to datetime first (if not already done)
    df[col_name] = pd.to_datetime(df[col_name], errors='coerce', format='%Y/%m/%d %H:%M:%S')

    # Extract the Date (yyyy/mm/dd) and Time (hh:mm:ss)
    df['Date'] = df[col_name].dt.date
    df['Call Start Time'] = df[col_name].dt.strftime('%H:%M:%S')

    return df

# Apply the function to each dataframe
# new_voiso = split_date_time(new_voiso, 'Date and time')
new_string = split_date_time(new_string, 'Start time')

# Verify the changes
print(new_qconn[['Date', 'Call Start Time']].head())
# print(new_voiso[['Date', 'Call Start Time']].head())
print(new_Know[['Date', 'Call Start Time']].head())
print(new_string[['Date', 'Call Start Time']].head())


Empty DataFrame
Columns: [Date, Call Start Time]
Index: []
         Date Call Start Time
0  2026-03-30        21:50:48
1  2026-03-30        21:05:23
2  2026-03-30        20:32:48
3  2026-03-30        20:17:31
4  2026-03-30        20:16:10
         Date Call Start Time
0  2026-03-30        19:30:07
1  2026-03-30        19:27:23
2  2026-03-30        18:53:04
3  2026-03-30        18:29:18
4  2026-03-30        18:28:17


C:\Users\Akhil\AppData\Local\Temp\ipykernel_13804\1294338338.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col_name] = pd.to_datetime(df[col_name], errors='coerce', format='%Y/%m/%d %H:%M:%S')
C:\Users\Akhil\AppData\Local\Temp\ipykernel_13804\1294338338.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Date'] = df[col_name].dt.date
C:\Users\Akhil\AppData\Local\Temp\ipykernel_13804\1294338338.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try 

In [31]:
new_qconn

,Date time,Agent Mobile,Call Event,Transfer Duration,Duration,User Mobile,Date,Call Start Time


### copy of main dfs

In [32]:
tata_copy = new_tata.copy()
know_copy = new_Know.copy()
# voiso_copy = new_voiso.copy()
qconn_copy = new_qconn.copy()
string_copy = new_string.copy()

### insert source 

In [33]:
tata_copy['Source'] = 'Tata'
know_copy['Source'] = 'Knowlarity'
# voiso_copy['Source'] = 'Voiso'
qconn_copy['Source'] = 'Qkonnect'
string_copy['Source'] = 'Stringee'


In [34]:
print(tata_copy.dtypes)
print(string_copy['Date'].unique())

Call Start Date                   object
Connected to Agent                object
Call Status                       object
Answer Duration (HH:MM:SS)        object
Hold Duration (HH:MM:SS)          object
Total Call Duration (HH:MM:SS)    object
Call Start Time                   object
Customer Number                    int64
Source                            object
dtype: object
[datetime.date(2026, 3, 30)]


### Rename Columns

In [35]:
tata_copy.rename(columns={
    'Call Start Date': 'Date',
    'Connected to Agent': 'Dialer Name',
    'Customer Number' : 'Number',
    'Call Status': 'Call Status',
    'Answer Duration (HH:MM:SS)': 'Talk Time',
    'Hold Duration (HH:MM:SS)': 'Hold Time',
    'Total Call Duration (HH:MM:SS)': 'Total Call Duration',
    'Call Start Time': 'Call Start Time'
}, inplace=True)

know_copy.rename(columns={
    'Date': 'Date',
    'Agent Name': 'Dialer Name',
    'Customer': 'Number',
    'Call Status': 'Call Status',
    'Talk Time (hh:mm:ss)': 'Talk Time',
    'Hold Time (hh:mm:ss)': 'Hold Time',
    'Total Call Duration (hh:mm:ss)': 'Total Call Duration',
    'Call Start Time': 'Call Start Time'
}, inplace=True)

# voiso_copy.rename(columns={
#     'Date': 'Date',
#     'Agent(s)': 'Dialer Name',
#     'DNIS/To': 'Number',
#     'Disposition': 'Call Status',
#     'Talk time': 'Talk Time',
#     'Duration': 'Total Call Duration',
#     'Call Start Time': 'Call Start Time'
# }, inplace=True)

qconn_copy.rename(columns={
    'Date': 'Date',
    'Agent Mobile': 'Dialer Name',
    'User Mobile': 'Number',
    'Call Event': 'Call Status',
    'Transfer Duration': 'Talk Time',
    'Duration': 'Total Call Duration',
    'Call Start Time': 'Call Start Time'
}, inplace=True)

string_copy.rename(columns={
    'Date': 'Date',    
    'Account': 'Dialer Name',
    'Customer number': 'Number',
    'Call status': 'Call Status',
    'Answer duration': 'Talk Time',
    'Hold duration': 'Hold Time',
    'Total Duration': 'Total Call Duration',
    'Call Start Time': 'Call Start Time'
}, inplace=True)



In [36]:
print(qconn_copy.shape)
# print(voiso_copy.shape)
print(know_copy.shape)  
print(tata_copy.shape)
print(string_copy.shape)

(0, 9)
(1334, 10)
(12919, 9)
(1619, 10)


In [37]:
print(string_copy['Date'].unique())
print(tata_copy['Date'].unique())
print(know_copy['Date'].unique())
# print(voiso_copy['Date'].unique())
print(qconn_copy['Date'].unique())


[datetime.date(2026, 3, 30)]
['2026-03-30']
[datetime.date(2026, 3, 30)]
[]


In [38]:
tata_copy.shape, know_copy.shape, qconn_copy.shape,string_copy.shape

((12919, 9), (1334, 10), (0, 9), (1619, 10))

In [39]:

# Select only the required columns from each dataframe
tata_selected = tata_copy[[ 'Source','Date', 'Dialer Name','Number', 'Call Status','Call Start Time','Total Call Duration', 'Talk Time', 'Hold Time']]
know_selected = know_copy[[ 'Source','Date', 'Dialer Name','Number' ,'Call Status', 'Call Start Time','Total Call Duration','Talk Time', 'Hold Time']]
# voiso_selected = voiso_copy[[ 'Source','Date', 'Dialer Name', 'Number','Call Status','Call Start Time','Total Call Duration', 'Talk Time']]
qconn_selected = qconn_copy[['Source','Date', 'Dialer Name','Number', 'Call Status','Call Start Time','Total Call Duration', 'Talk Time']]
string_selected = string_copy[['Source','Date', 'Dialer Name','Number', 'Call Status','Call Start Time','Total Call Duration', 'Talk Time', 'Hold Time']]


# Now concatenate
combined = pd.concat([tata_selected, know_selected, qconn_selected, string_selected], ignore_index=True)

In [40]:
combined

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time
0,Tata,2026-03-30,NaN,919653040864,Missed,21:19:52,00:00:05,00:00:00,00:00:00
1,Tata,2026-03-30,NaN,918585916957,Missed,21:11:49,00:00:05,00:00:00,00:00:00
2,Tata,2026-03-30,shiv,919691191881,Missed,20:46:32,00:00:21,00:00:00,00:00:00
3,Tata,2026-03-30,251 Chandan,919334658987,Missed,20:28:00,00:00:06,00:00:00,00:00:00
4,Tata,2026-03-30,251 Chandan,919125308105,Missed,20:27:09,00:00:08,00:00:00,00:00:00
...,...,...,...,...,...,...,...,...,...
15867,Stringee,2026-03-30,hritika@nexora.live,917559337579,Missed,09:42:31,0 days 00:00:44,00:00:00,00:00:00
15868,Stringee,2026-03-30,hritika@nexora.live,919131943813,Missed,09:41:30,0 days 00:00:27,00:00:00,00:00:00
15869,Stringee,2026-03-30,kawaljit.k@nexora.live,919999495915,Answered,09:40:40,0 days 00:01:19,00:00:49,00:00:00
15870,Stringee,2026-03-30,hritika@nexora.live,919140107280,Missed,09:40:18,0 days 00:00:26,00:00:00,00:00:00


In [41]:
print(combined.isnull().sum())
print(combined.shape)

Source                    0
Date                      0
Dialer Name            1345
Number                    0
Call Status               0
Call Start Time           0
Total Call Duration       0
Talk Time                 0
Hold Time                 0
dtype: int64
(15872, 9)


## Check_point_2

In [42]:
# Filter rows where Date is null
null_date_entries = combined[combined['Date'].isnull()]

# Display the filtered DataFrame
null_date_entries


,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time


In [43]:
combined['Date'].unique()

array(['2026-03-30', datetime.date(2026, 3, 30)], dtype=object)

In [44]:
combined_df =combined.copy()

In [45]:
# Refine the regex to only remove unwanted patterns
combined_df['Dialer Name'] = combined_df['Dialer Name'].str.replace(r"\s*\([^)]*\)|@.*|;.*", "", regex=True)

# Fill missing 'Dialer Name' values with their original values if they were numeric
combined_df['Dialer Name'] = combined_df['Dialer Name'].fillna(combined['Dialer Name'])

In [46]:
# Replace NaN in 'Hold time' with '00:00:00'
combined_df['Hold Time'] = combined_df['Hold Time'].fillna('00:00:00')

combined_df

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time
0,Tata,2026-03-30,NaN,919653040864,Missed,21:19:52,00:00:05,00:00:00,00:00:00
1,Tata,2026-03-30,NaN,918585916957,Missed,21:11:49,00:00:05,00:00:00,00:00:00
2,Tata,2026-03-30,shiv,919691191881,Missed,20:46:32,00:00:21,00:00:00,00:00:00
3,Tata,2026-03-30,251 Chandan,919334658987,Missed,20:28:00,00:00:06,00:00:00,00:00:00
4,Tata,2026-03-30,251 Chandan,919125308105,Missed,20:27:09,00:00:08,00:00:00,00:00:00
...,...,...,...,...,...,...,...,...,...
15867,Stringee,2026-03-30,hritika,917559337579,Missed,09:42:31,0 days 00:00:44,00:00:00,00:00:00
15868,Stringee,2026-03-30,hritika,919131943813,Missed,09:41:30,0 days 00:00:27,00:00:00,00:00:00
15869,Stringee,2026-03-30,kawaljit.k,919999495915,Answered,09:40:40,0 days 00:01:19,00:00:49,00:00:00
15870,Stringee,2026-03-30,hritika,919140107280,Missed,09:40:18,0 days 00:00:26,00:00:00,00:00:00


In [47]:
A = combined_df.copy()

In [48]:
# Remove rows where 'Dialer Name' is null
A1 = A[A['Dialer Name'].notnull()]

# Verify the result
print(f"Number of rows after removing null 'Dialer Name': {len(A1)}")

Number of rows after removing null 'Dialer Name': 14527


In [49]:
A1[A1['Dialer Name'].str.contains('223', na=False)]['Dialer Name'].unique()

array(['223 abhishek'], dtype=object)

In [50]:
A1.describe()

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time
count,14527,14527,14527,14527,14527,14527,14527,14527,14527
unique,3,2,68,12313,2,11989,619,451,10
top,Tata,2026-03-30,272 Sangita,919819182234,Missed,18:18:04,00:00:31,00:00:00,00:00:00
freq,11714,11714,368,13,11748,6,1764,10807,13245


In [51]:
# Check for null values in the 'Dialer Name' column
null_dialer_name_count = combined_df['Dialer Name'].isnull().sum()

# Print the result
print(f"Number of null values in 'Dialer Name': {null_dialer_name_count}")

Number of null values in 'Dialer Name': 1345


In [52]:
A1

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time
2,Tata,2026-03-30,shiv,919691191881,Missed,20:46:32,00:00:21,00:00:00,00:00:00
3,Tata,2026-03-30,251 Chandan,919334658987,Missed,20:28:00,00:00:06,00:00:00,00:00:00
4,Tata,2026-03-30,251 Chandan,919125308105,Missed,20:27:09,00:00:08,00:00:00,00:00:00
5,Tata,2026-03-30,251 Chandan,919356692250,Missed,20:26:07,00:00:06,00:00:00,00:00:00
6,Tata,2026-03-30,251 Chandan,918421727405,Missed,20:25:31,00:00:03,00:00:00,00:00:00
...,...,...,...,...,...,...,...,...,...
15867,Stringee,2026-03-30,hritika,917559337579,Missed,09:42:31,0 days 00:00:44,00:00:00,00:00:00
15868,Stringee,2026-03-30,hritika,919131943813,Missed,09:41:30,0 days 00:00:27,00:00:00,00:00:00
15869,Stringee,2026-03-30,kawaljit.k,919999495915,Answered,09:40:40,0 days 00:01:19,00:00:49,00:00:00
15870,Stringee,2026-03-30,hritika,919140107280,Missed,09:40:18,0 days 00:00:26,00:00:00,00:00:00


## Check_point_3

In [53]:
unique_dates_per_source = A1.groupby('Source')['Date'].unique()
print(unique_dates_per_source)

Source
Knowlarity    [2026-03-30]
Stringee      [2026-03-30]
Tata          [2026-03-30]
Name: Date, dtype: object


In [54]:
A1

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time
2,Tata,2026-03-30,shiv,919691191881,Missed,20:46:32,00:00:21,00:00:00,00:00:00
3,Tata,2026-03-30,251 Chandan,919334658987,Missed,20:28:00,00:00:06,00:00:00,00:00:00
4,Tata,2026-03-30,251 Chandan,919125308105,Missed,20:27:09,00:00:08,00:00:00,00:00:00
5,Tata,2026-03-30,251 Chandan,919356692250,Missed,20:26:07,00:00:06,00:00:00,00:00:00
6,Tata,2026-03-30,251 Chandan,918421727405,Missed,20:25:31,00:00:03,00:00:00,00:00:00
...,...,...,...,...,...,...,...,...,...
15867,Stringee,2026-03-30,hritika,917559337579,Missed,09:42:31,0 days 00:00:44,00:00:00,00:00:00
15868,Stringee,2026-03-30,hritika,919131943813,Missed,09:41:30,0 days 00:00:27,00:00:00,00:00:00
15869,Stringee,2026-03-30,kawaljit.k,919999495915,Answered,09:40:40,0 days 00:01:19,00:00:49,00:00:00
15870,Stringee,2026-03-30,hritika,919140107280,Missed,09:40:18,0 days 00:00:26,00:00:00,00:00:00


In [55]:
import re

def normalize_talk_time(talk_time):
    # If the value is in seconds (only digits), convert it to hh:mm:ss
    if re.match(r"^\d+$", str(talk_time)):
        seconds = int(talk_time)
        hours = seconds // 3600
        minutes = (seconds % 3600) // 60
        seconds = seconds % 60
        return f"{hours:02}:{minutes:02}:{seconds:02}"
    elif re.match(r"^\d+:\d+:\d+$", str(talk_time)):
        parts = talk_time.split(":")
        hours = int(parts[0])
        minutes = int(parts[1])
        seconds = int(parts[2])
        return f"{hours:02}:{minutes:02}:{seconds:02}"
    else:
        # Return the value as-is if it doesn't match expected formats
        return talk_time

# Apply the normalization function to the 'Talk Time' column
A1['Talk Time'] = A1['Talk Time'].apply(normalize_talk_time)

# Apply the normalization function to the 'Total Call Duration' column
A1['Total Call Duration'] = A1['Total Call Duration'].apply(normalize_talk_time)


print(A1[['Talk Time', 'Total Call Duration']].head())

  Talk Time Total Call Duration
2  00:00:00            00:00:21
3  00:00:00            00:00:06
4  00:00:00            00:00:08
5  00:00:00            00:00:06
6  00:00:00            00:00:03


C:\Users\Akhil\AppData\Local\Temp\ipykernel_13804\4084048447.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  A1['Talk Time'] = A1['Talk Time'].apply(normalize_talk_time)
C:\Users\Akhil\AppData\Local\Temp\ipykernel_13804\4084048447.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  A1['Total Call Duration'] = A1['Total Call Duration'].apply(normalize_talk_time)


In [56]:
unique_talk_time_formats = A1['Talk Time'].unique()

print(f"Unique formats in 'Talk Time': {unique_talk_time_formats}")

Unique formats in 'Talk Time': ['00:00:00' '00:00:33' '00:00:19' '00:01:06' '00:00:35' '00:00:16'
 '00:00:41' '00:00:39' '00:00:51' '00:00:24' '00:01:45' '00:00:36'
 '00:00:31' '00:00:28' '00:00:30' '00:03:00' '00:02:23' '00:00:03'
 '00:00:02' '00:00:06' '00:03:21' '00:01:49' '00:04:29' '00:01:25'
 '00:00:14' '00:00:05' '00:00:01' '00:10:04' '00:06:14' '00:00:15'
 '00:00:04' '00:02:18' '00:02:08' '00:00:18' '00:03:35' '00:00:22'
 '00:03:57' '00:00:13' '00:00:12' '00:00:40' '00:00:38' '00:00:09'
 '00:00:08' '00:05:02' '00:05:20' '00:00:21' '00:00:10' '00:06:43'
 '00:01:17' '00:00:52' '00:00:50' '00:00:07' '00:03:24' '00:00:23'
 '00:00:47' '00:02:04' '00:02:26' '00:00:27' '00:03:55' '00:04:43'
 '00:02:09' '00:00:20' '00:00:53' '00:00:11' '00:02:50' '00:01:20'
 '00:01:11' '00:04:58' '00:00:29' '00:03:03' '00:01:12' '00:00:57'
 '00:01:36' '00:12:56' '00:13:49' '00:00:26' '00:20:06' '00:00:44'
 '00:03:30' '00:05:14' '00:00:17' '00:06:09' '00:03:40' '00:02:02'
 '00:00:58' '00:00:55' '00:07:2

#### Unique call status (AOI)

In [57]:
A1['Call Status'].unique()

array(['Missed', 'Answered'], dtype=object)

In [58]:
A1['Call Status'] = A1['Call Status'].apply(
    lambda x: 'connected' if str(x).lower() == 'answered' else 'not connected'
)

C:\Users\Akhil\AppData\Local\Temp\ipykernel_13804\1189403410.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  A1['Call Status'] = A1['Call Status'].apply(


In [59]:
A1

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time
2,Tata,2026-03-30,shiv,919691191881,not connected,20:46:32,00:00:21,00:00:00,00:00:00
3,Tata,2026-03-30,251 Chandan,919334658987,not connected,20:28:00,00:00:06,00:00:00,00:00:00
4,Tata,2026-03-30,251 Chandan,919125308105,not connected,20:27:09,00:00:08,00:00:00,00:00:00
5,Tata,2026-03-30,251 Chandan,919356692250,not connected,20:26:07,00:00:06,00:00:00,00:00:00
6,Tata,2026-03-30,251 Chandan,918421727405,not connected,20:25:31,00:00:03,00:00:00,00:00:00
...,...,...,...,...,...,...,...,...,...
15867,Stringee,2026-03-30,hritika,917559337579,not connected,09:42:31,0 days 00:00:44,00:00:00,00:00:00
15868,Stringee,2026-03-30,hritika,919131943813,not connected,09:41:30,0 days 00:00:27,00:00:00,00:00:00
15869,Stringee,2026-03-30,kawaljit.k,919999495915,connected,09:40:40,0 days 00:01:19,00:00:49,00:00:00
15870,Stringee,2026-03-30,hritika,919140107280,not connected,09:40:18,0 days 00:00:26,00:00:00,00:00:00


In [60]:
len(A1['Dialer Name'].unique())

68

In [61]:
A1['Dialer Name'].unique()

array(['shiv', '251 Chandan', '580 Riddhi', '328 Anuj', '223 abhishek',
       '88 Waman', '614 Irfan', '599 Aamir', '272 Sangita', '569 Maaz',
       '577 Saurabh', '600 Yatin', '537 Mitali', 'darshini', 'adharv',
       'reeya', '502 Fahrukh', '597 Suresh', 'madhur', '552 Sonal',
       '497 Anjali', '596 Shambu', 'mohit', 'vaibhav', 'amrit', 'aaryan',
       '612 Sandeep', '498 Ankita', '18 PRINCE', 'shahid', 'haresh',
       '475 Sushil', 'musa', '584 Sneha', '539 Divya', '592 Aslam',
       'alina', '265 Amrut', 'ritesh', '259 Khushboo', 'sikander',
       '258 shivansh', 'kanika', '210 Sankalp', 'sana', '264  Lakshmi',
       '214 Shubham', '514 Tarun', 'zaid', '298 Sujal', 'Rajinder',
       'siddhesh', 'deepa', '01Monica ', '339ABHINAV ', '13Ashutosh ',
       '18Prince ', '124Sachin ', 'myron.f', 'arif.amin', 'asad.u',
       'esha', 'robin', 'ashpreet.k', 'kawaljit.k', 'dhanraj', 'pooja.k',
       'hritika'], dtype=object)

In [62]:
# Count the occurrences of each unique 'Dialer Name' and sort from highest to lowest
dialer_name_counts = A1['Dialer Name'].value_counts().sort_values(ascending=False)

print(dialer_name_counts)

Dialer Name
272 Sangita     368
497 Anjali      358
223 abhishek    356
592 Aslam       355
13Ashutosh      340
               ... 
Rajinder         80
dhanraj          67
esha             57
18 PRINCE        31
robin            30
Name: count, Length: 68, dtype: int64


In [63]:

combined_df_1 = A1[~A1['Dialer Name'].isin([None, '---'])]

combined_df_1.reset_index(drop=True, inplace=True)

combined_df_1

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time
0,Tata,2026-03-30,shiv,919691191881,not connected,20:46:32,00:00:21,00:00:00,00:00:00
1,Tata,2026-03-30,251 Chandan,919334658987,not connected,20:28:00,00:00:06,00:00:00,00:00:00
2,Tata,2026-03-30,251 Chandan,919125308105,not connected,20:27:09,00:00:08,00:00:00,00:00:00
3,Tata,2026-03-30,251 Chandan,919356692250,not connected,20:26:07,00:00:06,00:00:00,00:00:00
4,Tata,2026-03-30,251 Chandan,918421727405,not connected,20:25:31,00:00:03,00:00:00,00:00:00
...,...,...,...,...,...,...,...,...,...
14522,Stringee,2026-03-30,hritika,917559337579,not connected,09:42:31,0 days 00:00:44,00:00:00,00:00:00
14523,Stringee,2026-03-30,hritika,919131943813,not connected,09:41:30,0 days 00:00:27,00:00:00,00:00:00
14524,Stringee,2026-03-30,kawaljit.k,919999495915,connected,09:40:40,0 days 00:01:19,00:00:49,00:00:00
14525,Stringee,2026-03-30,hritika,919140107280,not connected,09:40:18,0 days 00:00:26,00:00:00,00:00:00


In [64]:
combined_df_1

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time
0,Tata,2026-03-30,shiv,919691191881,not connected,20:46:32,00:00:21,00:00:00,00:00:00
1,Tata,2026-03-30,251 Chandan,919334658987,not connected,20:28:00,00:00:06,00:00:00,00:00:00
2,Tata,2026-03-30,251 Chandan,919125308105,not connected,20:27:09,00:00:08,00:00:00,00:00:00
3,Tata,2026-03-30,251 Chandan,919356692250,not connected,20:26:07,00:00:06,00:00:00,00:00:00
4,Tata,2026-03-30,251 Chandan,918421727405,not connected,20:25:31,00:00:03,00:00:00,00:00:00
...,...,...,...,...,...,...,...,...,...
14522,Stringee,2026-03-30,hritika,917559337579,not connected,09:42:31,0 days 00:00:44,00:00:00,00:00:00
14523,Stringee,2026-03-30,hritika,919131943813,not connected,09:41:30,0 days 00:00:27,00:00:00,00:00:00
14524,Stringee,2026-03-30,kawaljit.k,919999495915,connected,09:40:40,0 days 00:01:19,00:00:49,00:00:00
14525,Stringee,2026-03-30,hritika,919140107280,not connected,09:40:18,0 days 00:00:26,00:00:00,00:00:00


In [65]:
ref_d = pd.read_excel('Team_tradex.xlsx')

In [66]:
ref_d.nunique()

Dialer Name      207
Dialer             6
Email            114
Employee code    114
Full Name        114
Pool              10
TL                 7
Vertical           1
dtype: int64

In [67]:
ref= ref_d.copy()

In [68]:
ref = ref[~ref['Email'].str.contains('inactive', case=False, na=False)]

In [69]:
ref

,Dialer Name,Dialer,Email,Employee code,Full Name,Pool,TL,Vertical
0,223Abhishek,Knowlarity,abhishek.k@nexora.live,E-1266,Abhishek Kumar,Meta,Radhika,TradeX
1,223 Abhishek,Tata,abhishek.k@nexora.live,E-1266,Abhishek Kumar,Meta,Radhika,TradeX
2,adharv,Tata,adharv@nexora.live,E-1460,Anuj Saini,Dialer,Rehan,TradeX
3,412 Anuj,Tata,adharv@nexora.live,E-1460,Anuj Saini,Dialer,Rehan,TradeX
4,advit,Tata,advit@nexora.live,E-1456,Rohit Goyal,Dialer,Rehan,TradeX
...,...,...,...,...,...,...,...,...
202,Mohit,Tata,mohit@nexora.live,E-2133,Mohit,OJT 1,Rehan,TradeX
203,Alina,Tata,alina@nexora.live,E-2135,Alina,OJT 1,Rehan,TradeX
204,Siddhesh,Tata,siddhesh@nexora.live,E-2137,Siddhesh,OJT 1,Rehan,TradeX
205,Reeya,Tata,reeya@nexora.live,E-2126,Reeya,OJT 1,Rehan,TradeX


## Checkpoint_4

In [70]:
ref.nunique()

Dialer Name      207
Dialer             6
Email            114
Employee code    114
Full Name        114
Pool              10
TL                 7
Vertical           1
dtype: int64

In [71]:
for df in [ref, combined_df_1]:
    df['Dialer Name'] = (
        df['Dialer Name']
        .astype(str)  # Convert all values to strings
        .str.replace(r'\s+', ' ', regex=True)  # Replace multiple spaces with a single space
        .str.strip()  # Remove leading and trailing spaces
        .str.replace(r'@.*', '', regex=True)  # Remove everything from and after '@'
        .str.replace(r'\(.*', '', regex=True)  # Remove everything from and after '('
        .str.strip()  # Remove any trailing spaces left after replacements
        .str.lower()  # Convert all text to lowercase
    )


In [72]:
ref.shape

(207, 8)

In [73]:
ref.isnull().sum()

Dialer Name      0
Dialer           0
Email            0
Employee code    0
Full Name        0
Pool             2
TL               2
Vertical         0
dtype: int64

In [74]:
dup = ref[ref.duplicated(subset=['Dialer Name','Email','Dialer'], keep=False)]
dup

,Dialer Name,Dialer,Email,Employee code,Full Name,Pool,TL,Vertical


In [75]:
ref = ref.drop_duplicates(subset=['Dialer Name','Email','Dialer'])

In [76]:
ref.rename(columns={'Email': 'CRM ID'}, inplace=True)

In [77]:
ref

,Dialer Name,Dialer,CRM ID,Employee code,Full Name,Pool,TL,Vertical
0,223abhishek,Knowlarity,abhishek.k@nexora.live,E-1266,Abhishek Kumar,Meta,Radhika,TradeX
1,223 abhishek,Tata,abhishek.k@nexora.live,E-1266,Abhishek Kumar,Meta,Radhika,TradeX
2,adharv,Tata,adharv@nexora.live,E-1460,Anuj Saini,Dialer,Rehan,TradeX
3,412 anuj,Tata,adharv@nexora.live,E-1460,Anuj Saini,Dialer,Rehan,TradeX
4,advit,Tata,advit@nexora.live,E-1456,Rohit Goyal,Dialer,Rehan,TradeX
...,...,...,...,...,...,...,...,...
202,mohit,Tata,mohit@nexora.live,E-2133,Mohit,OJT 1,Rehan,TradeX
203,alina,Tata,alina@nexora.live,E-2135,Alina,OJT 1,Rehan,TradeX
204,siddhesh,Tata,siddhesh@nexora.live,E-2137,Siddhesh,OJT 1,Rehan,TradeX
205,reeya,Tata,reeya@nexora.live,E-2126,Reeya,OJT 1,Rehan,TradeX


In [78]:
# Ensure 'Dialer Name' in both dataframes is treated as a string
combined_df_1['Dialer Name'] = combined_df_1['Dialer Name'].astype(str)
ref['Dialer Name'] = ref['Dialer Name'].astype(str)

# Merge the dataframes on 'Dialer Name'
combined_df_2 = combined_df_1.merge(ref, how='left', left_on='Dialer Name', right_on='Dialer Name')


combined_df_2


,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time,Dialer,CRM ID,Employee code,Full Name,Pool,TL,Vertical
0,Tata,2026-03-30,shiv,919691191881,not connected,20:46:32,00:00:21,00:00:00,00:00:00,Tata,shiv@nexora.live,E-2121,Shiv,OJT 1,Rehan,TradeX
1,Tata,2026-03-30,251 chandan,919334658987,not connected,20:28:00,00:00:06,00:00:00,00:00:00,Tata,chandan.y@nexora.live,E-1274,Chandan Yadav,Meta,Radhika,TradeX
2,Tata,2026-03-30,251 chandan,919125308105,not connected,20:27:09,00:00:08,00:00:00,00:00:00,Tata,chandan.y@nexora.live,E-1274,Chandan Yadav,Meta,Radhika,TradeX
3,Tata,2026-03-30,251 chandan,919356692250,not connected,20:26:07,00:00:06,00:00:00,00:00:00,Tata,chandan.y@nexora.live,E-1274,Chandan Yadav,Meta,Radhika,TradeX
4,Tata,2026-03-30,251 chandan,918421727405,not connected,20:25:31,00:00:03,00:00:00,00:00:00,Tata,chandan.y@nexora.live,E-1274,Chandan Yadav,Meta,Radhika,TradeX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14718,Stringee,2026-03-30,kawaljit.k,919999495915,connected,09:40:40,0 days 00:01:19,00:00:49,00:00:00,Stringee,kawaljit.k@nexora.live,E-1237,Kawaljit Kaur,Meta,Radhika,TradeX
14719,Stringee,2026-03-30,hritika,919140107280,not connected,09:40:18,0 days 00:00:26,00:00:00,00:00:00,Voiso,hritika@nexora.live,E-1637,Astha Tyagi,Pull Back,Parth,TradeX
14720,Stringee,2026-03-30,hritika,919140107280,not connected,09:40:18,0 days 00:00:26,00:00:00,00:00:00,Stringee,hritika@nexora.live,E-1637,Astha Tyagi,Pull Back,Parth,TradeX
14721,Stringee,2026-03-30,hritika,919888560718,not connected,09:39:04,0 days 00:00:28,00:00:00,00:00:00,Voiso,hritika@nexora.live,E-1637,Astha Tyagi,Pull Back,Parth,TradeX


In [79]:
# Filter rows where 'CRM ID' is null
crm_id_null_df = combined_df_2[combined_df_2['CRM ID'].isnull()]


In [80]:

crm_id_null_df['Dialer Name'].unique()

array(['robin'], dtype=object)

In [81]:
crm_id_null_df

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time,Dialer,CRM ID,Employee code,Full Name,Pool,TL,Vertical
13633,Stringee,2026-03-30,robin,919915070291,not connected,22:20:35,0 days 00:00:35,00:00:00,00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13634,Stringee,2026-03-30,robin,919491056789,connected,21:35:52,0 days 00:02:15,00:02:07,00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13635,Stringee,2026-03-30,robin,918286986544,connected,17:57:26,0 days 00:03:13,00:02:51,00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13643,Stringee,2026-03-30,robin,919979145452,connected,17:34:06,0 days 00:04:32,00:04:22,00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13644,Stringee,2026-03-30,robin,917893957617,connected,17:15:14,0 days 00:01:21,00:01:14,00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13645,Stringee,2026-03-30,robin,917893957617,connected,17:14:45,0 days 00:00:24,00:00:18,00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13646,Stringee,2026-03-30,robin,917893957617,not connected,17:14:03,0 days 00:00:38,00:00:00,00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13657,Stringee,2026-03-30,robin,919355045400,connected,16:59:39,0 days 00:00:53,00:00:36,00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13679,Stringee,2026-03-30,robin,919876999813,not connected,16:13:13,0 days 00:00:48,00:00:00,00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13683,Stringee,2026-03-30,robin,919855861113,not connected,16:09:01,0 days 00:00:41,00:00:00,00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## check_point_5

In [82]:
source = crm_id_null_df.groupby('Source')['Dialer Name'].unique()
source

Source
Stringee    [robin]
Name: Dialer Name, dtype: object

In [83]:
crm_id_null_df.shape

(30, 16)

## Raw data

In [84]:
combined_df_2

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time,Dialer,CRM ID,Employee code,Full Name,Pool,TL,Vertical
0,Tata,2026-03-30,shiv,919691191881,not connected,20:46:32,00:00:21,00:00:00,00:00:00,Tata,shiv@nexora.live,E-2121,Shiv,OJT 1,Rehan,TradeX
1,Tata,2026-03-30,251 chandan,919334658987,not connected,20:28:00,00:00:06,00:00:00,00:00:00,Tata,chandan.y@nexora.live,E-1274,Chandan Yadav,Meta,Radhika,TradeX
2,Tata,2026-03-30,251 chandan,919125308105,not connected,20:27:09,00:00:08,00:00:00,00:00:00,Tata,chandan.y@nexora.live,E-1274,Chandan Yadav,Meta,Radhika,TradeX
3,Tata,2026-03-30,251 chandan,919356692250,not connected,20:26:07,00:00:06,00:00:00,00:00:00,Tata,chandan.y@nexora.live,E-1274,Chandan Yadav,Meta,Radhika,TradeX
4,Tata,2026-03-30,251 chandan,918421727405,not connected,20:25:31,00:00:03,00:00:00,00:00:00,Tata,chandan.y@nexora.live,E-1274,Chandan Yadav,Meta,Radhika,TradeX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14718,Stringee,2026-03-30,kawaljit.k,919999495915,connected,09:40:40,0 days 00:01:19,00:00:49,00:00:00,Stringee,kawaljit.k@nexora.live,E-1237,Kawaljit Kaur,Meta,Radhika,TradeX
14719,Stringee,2026-03-30,hritika,919140107280,not connected,09:40:18,0 days 00:00:26,00:00:00,00:00:00,Voiso,hritika@nexora.live,E-1637,Astha Tyagi,Pull Back,Parth,TradeX
14720,Stringee,2026-03-30,hritika,919140107280,not connected,09:40:18,0 days 00:00:26,00:00:00,00:00:00,Stringee,hritika@nexora.live,E-1637,Astha Tyagi,Pull Back,Parth,TradeX
14721,Stringee,2026-03-30,hritika,919888560718,not connected,09:39:04,0 days 00:00:28,00:00:00,00:00:00,Voiso,hritika@nexora.live,E-1637,Astha Tyagi,Pull Back,Parth,TradeX


In [85]:
Dialers = combined_df_2[combined_df_2['CRM ID'].notnull() & combined_df_2['Talk Time'].notnull()].copy()

In [86]:
Dialers.isnull().sum()

Source                 0
Date                   0
Dialer Name            0
Number                 0
Call Status            0
Call Start Time        0
Total Call Duration    0
Talk Time              0
Hold Time              0
Dialer                 0
CRM ID                 0
Employee code          0
Full Name              0
Pool                   0
TL                     0
Vertical               0
dtype: int64

In [87]:
Dialers = Dialers.drop_duplicates(subset=['Number','Call Start Time'])


In [88]:
XXX = combined_df_2[combined_df_2['Date'].isnull()]
XXX

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time,Dialer,CRM ID,Employee code,Full Name,Pool,TL,Vertical


In [89]:
Dialers.dtypes

Source                 object
Date                   object
Dialer Name            object
Number                 object
Call Status            object
Call Start Time        object
Total Call Duration    object
Talk Time              object
Hold Time              object
Dialer                 object
CRM ID                 object
Employee code          object
Full Name              object
Pool                   object
TL                     object
Vertical               object
dtype: object

In [90]:
from datetime import timedelta

Dialers['Date'] = pd.to_datetime(Dialers['Date'], format='%Y-%m-%d', errors='coerce')
Dialers['Call Start Time'] = pd.to_datetime(
    Dialers['Date'].dt.strftime('%Y-%m-%d').fillna('1900-01-01') + ' ' + Dialers['Call Start Time'],
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce'
)

# Convert durations
Dialers['Total Call Duration'] = Dialers['Total Call Duration'].apply(
    lambda x: pd.to_timedelta(x) if isinstance(x, str) else pd.Timedelta(0)
)

# Sort
Dialers = Dialers.sort_values(by=['Date', 'CRM ID', 'Call Start Time']).reset_index(drop=True)

# Init columns
Dialers['Call Gap'] = 'No'
Dialers['Gap Duration'] = '00:00:00'

# Define working hours
start_time = pd.to_datetime('09:30:00').time()
end_time = pd.to_datetime('18:30:00').time()

# Loop through to calculate gaps
for i in range(1, len(Dialers)):
    same_crm = Dialers.loc[i, 'CRM ID'] == Dialers.loc[i - 1, 'CRM ID']
    same_date = Dialers.loc[i, 'Date'] == Dialers.loc[i - 1, 'Date']  

    if same_crm and same_date:
        current_time = Dialers.loc[i, 'Call Start Time'].time()
        previous_time = Dialers.loc[i - 1, 'Call Start Time'].time()

        previous_end = Dialers.loc[i - 1, 'Call Start Time'] + Dialers.loc[i - 1, 'Total Call Duration']
        gap_duration = Dialers.loc[i, 'Call Start Time'] - previous_end

        if gap_duration.total_seconds() < 0:
            gap_duration = timedelta(0)

        # Store formatted gap
        total_seconds = int(gap_duration.total_seconds())
        hours = total_seconds // 3600
        minutes = (total_seconds % 3600) // 60
        seconds = total_seconds % 60
        Dialers.loc[i, 'Gap Duration'] = f"{hours:02}:{minutes:02}:{seconds:02}"

        # Set 'Call Gap' only if within working hours
        if start_time <= current_time <= end_time and start_time <= previous_time <= end_time:
            Dialers.loc[i, 'Call Gap'] = 'Yes' if gap_duration > timedelta(minutes=1) else 'No'

Dialers



,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time,Dialer,CRM ID,Employee code,Full Name,Pool,TL,Vertical,Call Gap,Gap Duration
0,Tata,2026-03-30,599 aamir,917522088758,not connected,2026-03-30 10:20:18,0 days 00:00:01,00:00:00,00:00:00,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:00
1,Tata,2026-03-30,599 aamir,917854014859,not connected,2026-03-30 10:59:03,0 days 00:00:31,00:00:00,00:00:00,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,Yes,00:38:44
2,Tata,2026-03-30,599 aamir,919636559741,not connected,2026-03-30 11:01:20,0 days 00:00:31,00:00:00,00:00:00,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,Yes,00:01:46
3,Tata,2026-03-30,599 aamir,919719218429,not connected,2026-03-30 11:06:48,0 days 00:00:31,00:00:00,00:00:00,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,Yes,00:04:57
4,Tata,2026-03-30,599 aamir,919148696641,not connected,2026-03-30 11:07:29,0 days 00:00:04,00:00:00,00:00:00,Tata,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14474,Tata,2026-03-30,zaid,918387958268,not connected,2026-03-30 18:17:30,0 days 00:00:31,00:00:00,00:00:00,Tata,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,Yes,00:01:11
14475,Tata,2026-03-30,zaid,917385290403,not connected,2026-03-30 18:19:39,0 days 00:00:31,00:00:00,00:00:00,Tata,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,Yes,00:01:38
14476,Tata,2026-03-30,zaid,919572640835,not connected,2026-03-30 18:20:23,0 days 00:00:31,00:00:00,00:00:00,Tata,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,No,00:00:13
14477,Tata,2026-03-30,zaid,917008344146,not connected,2026-03-30 18:22:22,0 days 00:00:21,00:00:00,00:00:00,Tata,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,Yes,00:01:28


In [91]:
Dialers.isnull().sum()

Source                 0
Date                   0
Dialer Name            0
Number                 0
Call Status            0
Call Start Time        0
Total Call Duration    0
Talk Time              0
Hold Time              0
Dialer                 0
CRM ID                 0
Employee code          0
Full Name              0
Pool                   0
TL                     0
Vertical               0
Call Gap               0
Gap Duration           0
dtype: int64

In [92]:
import re
invalid_values = []

# Function to convert mixed formats to seconds
def to_seconds(value):
    try:
        if isinstance(value, pd.Timedelta):
            return int(value.total_seconds())  # Convert timedelta to seconds
        elif re.match(r"^\d{1,2}:\d{1,2}:\d{1,2}$", str(value)):
            parts = list(map(int, value.split(':')))
            while len(parts) < 3:
                parts.insert(0, 0)  
            return parts[0] * 3600 + parts[1] * 60 + parts[2]
        elif str(value).isdigit():
            return int(value)
        else:
            invalid_values.append(value)
            return value
    except Exception:
        invalid_values.append(value)
        return value

# Replace missing values with empty strings
Dialers['Talk Time'] = Dialers['Talk Time'].fillna('')
Dialers['Hold Time'] = Dialers['Hold Time'].fillna('')
Dialers['Total Call Duration'] = Dialers['Total Call Duration'].fillna('')
Dialers['Gap Duration'] = Dialers['Gap Duration'].fillna('')

# Convert Talk Time, Hold Time, and Total Call Duration to seconds
Dialers['Talk Time (seconds)'] = Dialers['Talk Time'].apply(to_seconds)
Dialers['Hold Time (seconds)'] = Dialers['Hold Time'].apply(to_seconds)
Dialers['Total Duration (seconds)'] = Dialers['Total Call Duration'].apply(to_seconds)
Dialers['Gap Duration (seconds)'] = Dialers['Gap Duration'].apply(to_seconds)

# Notify invalid values
if invalid_values:
    print("Invalid values found in 'Talk Time', 'Total Call Duration', or 'Hold Time':, or 'Call Gap Duration':")
    print(invalid_values)

In [93]:
Dialers.isnull().sum()

Source                      0
Date                        0
Dialer Name                 0
Number                      0
Call Status                 0
Call Start Time             0
Total Call Duration         0
Talk Time                   0
Hold Time                   0
Dialer                      0
CRM ID                      0
Employee code               0
Full Name                   0
Pool                        0
TL                          0
Vertical                    0
Call Gap                    0
Gap Duration                0
Talk Time (seconds)         0
Hold Time (seconds)         0
Total Duration (seconds)    0
Gap Duration (seconds)      0
dtype: int64

### Change


In [94]:
# df['Total_Duration'] = pd.to_timedelta(df['Total_Duration'], errors='coerce')

In [95]:
# Group by 'CRM ID' and 'Date' and calculate 
A = Dialers.groupby(['CRM ID', 'Date']).agg(
    Total_Dialed_Calls=('Call Status', 'count'),
    Unique_Dialed_Numbers=('Number', 'nunique'),
    Total_Connected_Calls=('Call Status', lambda x: (x == 'connected').sum()),
    Total_Number_of_Call_Gap=('Call Gap', lambda x: (x == 'Yes').sum()),
    Total_Call_GT_30=('Talk Time (seconds)', lambda x: ((Dialers.loc[x.index, 'Call Status'] == 'connected') & (x > 30)).sum()),
    Total_Duration=('Total Duration (seconds)', 'sum'),
    Total_Talk_Time=('Talk Time (seconds)', lambda x: x[Dialers.loc[x.index, 'Call Status'] == 'connected'].sum()),
    Total_Talk_Time_GT_30=('Talk Time (seconds)', lambda x: x[(Dialers.loc[x.index, 'Call Status'] == 'connected') & (x > 30)].sum()),
    Total_Connected_Hold_Time=('Hold Time (seconds)', lambda x: x[Dialers.loc[x.index, 'Call Status'] == 'connected'].sum()), 
    Total_Gap_Duration=('Gap Duration (seconds)', 'sum')
).reset_index()

# Fix: Subtract 1hr only if greater, else keep the original value
A['Total_Gap_Duration'] = A['Total_Gap_Duration'].apply(lambda x: x - 3600 if x > 3600 else x)


A['Avg_Gap_per_call'] = A['Total_Gap_Duration'] / A['Total_Dialed_Calls']

# New: Gap Duration After Leverage (give 45 seconds per call, subtract from actual gap used, add to 0 if negative)
A['Gap Duration After Leverage'] = (A['Total_Gap_Duration'] - (A['Total_Dialed_Calls'] * 45)).clip(lower=0)

# Recalculate Login Hours after updated gap
A['Login Hours'] = A['Total_Duration'] + A['Total_Gap_Duration'] + 3600
A['Login Hours'] = pd.to_timedelta(A['Login Hours'], unit='s')
A['Login Hours'] = A['Login Hours'].apply(lambda x: str(x).split()[-1])

# Convert 'Login Hours' string (hh:mm:ss) to timedelta
A['Login Hours (Timedelta)'] = pd.to_timedelta(A['Login Hours'])

# Defining attendance logic
def mark_attendance(td):
    if td < pd.Timedelta(hours=4, minutes=30):
        return 'Absent'
    elif td < pd.Timedelta(hours=6):
        return 'Half Day'
    elif td < pd.Timedelta(hours=8, minutes=30):
        return 'Warning'
    else:
        return 'Present'


A['Attendance'] = A['Login Hours (Timedelta)'].apply(mark_attendance)


A.drop(columns='Login Hours (Timedelta)', inplace=True)


A


,CRM ID,Date,Total_Dialed_Calls,Unique_Dialed_Numbers,Total_Connected_Calls,Total_Number_of_Call_Gap,Total_Call_GT_30,Total_Duration,Total_Talk_Time,Total_Talk_Time_GT_30,Total_Connected_Hold_Time,Total_Gap_Duration,Avg_Gap_per_call,Gap Duration After Leverage,Login Hours,Attendance
0,aamir@nexora.live,2026-03-30,247,221,26,39,3,3984,406,103,0,26606,107.716599,15491,09:29:50,Present
1,aaryan@nexora.live,2026-03-30,200,186,33,96,12,6425,1898,1626,0,23452,117.260000,14452,09:17:57,Present
2,abhinav.a@nexora.live,2026-03-30,215,190,41,95,8,5994,1003,643,0,25839,120.181395,16164,09:50:33,Present
3,abhishek.k@nexora.live,2026-03-30,356,229,55,57,33,14811,7256,6985,0,18942,53.207865,2922,10:22:33,Present
4,adharv@nexora.live,2026-03-30,291,230,49,83,13,9686,2997,2580,0,23370,80.309278,10275,10:10:56,Present
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,tarun@nexora.live,2026-03-30,122,114,14,46,4,3108,277,150,0,23558,193.098361,18068,08:24:26,Warning
62,vaibhav@nexora.live,2026-03-30,189,143,42,73,20,11264,7077,6788,0,18839,99.677249,10334,09:21:43,Present
63,waman@nexora.live,2026-03-30,222,170,95,83,51,13250,9051,8399,0,22305,100.472973,12315,10:52:35,Present
64,yatin@nexora.live,2026-03-30,222,218,11,60,2,2217,135,78,0,31363,141.274775,21373,10:19:40,Present


In [96]:
AB = A['Attendance'].unique()
AB

array(['Present', 'Warning', 'Absent'], dtype=object)

In [97]:
ref

,Dialer Name,Dialer,CRM ID,Employee code,Full Name,Pool,TL,Vertical
0,223abhishek,Knowlarity,abhishek.k@nexora.live,E-1266,Abhishek Kumar,Meta,Radhika,TradeX
1,223 abhishek,Tata,abhishek.k@nexora.live,E-1266,Abhishek Kumar,Meta,Radhika,TradeX
2,adharv,Tata,adharv@nexora.live,E-1460,Anuj Saini,Dialer,Rehan,TradeX
3,412 anuj,Tata,adharv@nexora.live,E-1460,Anuj Saini,Dialer,Rehan,TradeX
4,advit,Tata,advit@nexora.live,E-1456,Rohit Goyal,Dialer,Rehan,TradeX
...,...,...,...,...,...,...,...,...
202,mohit,Tata,mohit@nexora.live,E-2133,Mohit,OJT 1,Rehan,TradeX
203,alina,Tata,alina@nexora.live,E-2135,Alina,OJT 1,Rehan,TradeX
204,siddhesh,Tata,siddhesh@nexora.live,E-2137,Siddhesh,OJT 1,Rehan,TradeX
205,reeya,Tata,reeya@nexora.live,E-2126,Reeya,OJT 1,Rehan,TradeX


In [98]:
# Filter unique CRM ID and select specific columns
unique_crm_ref = ref.drop_duplicates(subset=['CRM ID'])[['CRM ID','Employee code', 'Full Name','Pool', 'TL','Vertical']]

unique_crm_ref

,CRM ID,Employee code,Full Name,Pool,TL,Vertical
0,abhishek.k@nexora.live,E-1266,Abhishek Kumar,Meta,Radhika,TradeX
2,adharv@nexora.live,E-1460,Anuj Saini,Dialer,Rehan,TradeX
4,advit@nexora.live,E-1456,Rohit Goyal,Dialer,Rehan,TradeX
6,amrut.s@nexora.live,E-1282,Amrut Shivaji,Pull back-WFH,Swathi,TradeX
7,ashutosh@nexora.live,E-1253,Ashutosh Singh,1,Saif,TradeX
...,...,...,...,...,...,...
201,kanika@nexora.live,E-2132,Kanika,OJT 1,Rehan,TradeX
202,mohit@nexora.live,E-2133,Mohit,OJT 1,Rehan,TradeX
203,alina@nexora.live,E-2135,Alina,OJT 1,Rehan,TradeX
204,siddhesh@nexora.live,E-2137,Siddhesh,OJT 1,Rehan,TradeX


In [99]:
# Extract unique dates from DataFrame A
all_dates = A['Date'].unique()

# Create a DataFrame with all CRM IDs from unique_crm_ref and all dates
date_crm_combinations = pd.MultiIndex.from_product(
    [unique_crm_ref['CRM ID'], all_dates],
    names=['CRM ID', 'Date']
).to_frame(index=False)



merged = date_crm_combinations.merge(
    A,
    how='left',
    on=['CRM ID', 'Date']
).fillna(0)  


In [100]:
merged

,CRM ID,Date,Total_Dialed_Calls,Unique_Dialed_Numbers,Total_Connected_Calls,Total_Number_of_Call_Gap,Total_Call_GT_30,Total_Duration,Total_Talk_Time,Total_Talk_Time_GT_30,Total_Connected_Hold_Time,Total_Gap_Duration,Avg_Gap_per_call,Gap Duration After Leverage,Login Hours,Attendance
0,abhishek.k@nexora.live,2026-03-30,356.0,229.0,55.0,57.0,33.0,14811.0,7256.0,6985.0,0.0,18942.0,53.207865,2922.0,10:22:33,Present
1,adharv@nexora.live,2026-03-30,291.0,230.0,49.0,83.0,13.0,9686.0,2997.0,2580.0,0.0,23370.0,80.309278,10275.0,10:10:56,Present
2,advit@nexora.live,2026-03-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0,0
3,amrut.s@nexora.live,2026-03-30,164.0,151.0,141.0,149.0,64.0,8691.0,6293.0,5394.0,0.0,20657.0,125.957317,13277.0,09:09:08,Present
4,ashutosh@nexora.live,2026-03-30,340.0,300.0,85.0,58.0,45.0,11904.0,5177.0,4712.0,0.0,18112.0,53.270588,2812.0,09:20:16,Present
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,kanika@nexora.live,2026-03-30,228.0,217.0,59.0,76.0,21.0,6665.0,1985.0,1472.0,0.0,21888.0,96.000000,11628.0,08:55:53,Present
110,mohit@nexora.live,2026-03-30,185.0,151.0,20.0,60.0,5.0,4362.0,470.0,196.0,0.0,24907.0,134.632432,16582.0,09:07:49,Present
111,alina@nexora.live,2026-03-30,224.0,200.0,39.0,69.0,21.0,8174.0,2969.0,2769.0,0.0,21137.0,94.361607,11057.0,09:08:31,Present
112,siddhesh@nexora.live,2026-03-30,206.0,201.0,36.0,27.0,3.0,3973.0,434.0,125.0,0.0,18599.0,90.286408,9329.0,07:16:12,Warning


In [101]:
# Outer-merge with unique_crm_ref to retain all CRM IDs
merged_df = unique_crm_ref.merge(
    merged,
    how='outer',
    on='CRM ID'
)

In [102]:
merged_df

,CRM ID,Employee code,Full Name,Pool,TL,Vertical,Date,Total_Dialed_Calls,Unique_Dialed_Numbers,Total_Connected_Calls,...,Total_Call_GT_30,Total_Duration,Total_Talk_Time,Total_Talk_Time_GT_30,Total_Connected_Hold_Time,Total_Gap_Duration,Avg_Gap_per_call,Gap Duration After Leverage,Login Hours,Attendance
0,Deepa.negi@nexora.live,E-1443,Suja Basnet,Meta,Radhika,TradeX,2026-03-30,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0,0
1,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,2026-03-30,247.0,221.0,26.0,...,3.0,3984.0,406.0,103.0,0.0,26606.0,107.716599,15491.0,09:29:50,Present
2,aaryan@nexora.live,E-2124,Aaryan,OJT 1,Rehan,TradeX,2026-03-30,200.0,186.0,33.0,...,12.0,6425.0,1898.0,1626.0,0.0,23452.0,117.260000,14452.0,09:17:57,Present
3,abhinav.a@nexora.live,E-1356,Abhinav,1,Saif,TradeX,2026-03-30,215.0,190.0,41.0,...,8.0,5994.0,1003.0,643.0,0.0,25839.0,120.181395,16164.0,09:50:33,Present
4,abhishek.k@nexora.live,E-1266,Abhishek Kumar,Meta,Radhika,TradeX,2026-03-30,356.0,229.0,55.0,...,33.0,14811.0,7256.0,6985.0,0.0,18942.0,53.207865,2922.0,10:22:33,Present
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,vandana@nexora.live,E - 1677,Vandana,Dialer,Rehan,TradeX,2026-03-30,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0,0
110,vicky.v@nexora.live,E-1167,Vishal,Customer care,Vikas,TradeX,2026-03-30,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0,0
111,waman@nexora.live,E-1252,Waman Avhad,1,Saif,TradeX,2026-03-30,222.0,170.0,95.0,...,51.0,13250.0,9051.0,8399.0,0.0,22305.0,100.472973,12315.0,10:52:35,Present
112,yatin@nexora.live,E-1906,Yatin,OJT,Saif,TradeX,2026-03-30,222.0,218.0,11.0,...,2.0,2217.0,135.0,78.0,0.0,31363.0,141.274775,21373.0,10:19:40,Present


In [103]:
# Replace NaN or missing values with 0
columns_to_format = ['Total_Dialed_Calls', 'Unique_Dialed_Numbers','Total_Connected_Calls','Total_Number_of_Call_Gap', 'Total_Call_GT_30','Total_Duration','Total_Talk_Time','Total_Talk_Time_GT_30','Total_Connected_Hold_Time','Total_Gap_Duration']
merged_df[columns_to_format] = merged_df[columns_to_format].fillna(0)

# Convert specified columns to integers
merged_df[columns_to_format] = merged_df[columns_to_format].astype(int)

# Reorder columns
formatted_df = merged_df[['Date', 'Pool', 'TL', 'CRM ID','Employee code', 'Full Name', 'Vertical'] + [col for col in merged_df.columns if col not in ['Date', 'Pool', 'TL', 'CRM ID','Employee code', 'Full Name', 'Vertical']]]

formatted_df

,Date,Pool,TL,CRM ID,Employee code,Full Name,Vertical,Total_Dialed_Calls,Unique_Dialed_Numbers,Total_Connected_Calls,...,Total_Call_GT_30,Total_Duration,Total_Talk_Time,Total_Talk_Time_GT_30,Total_Connected_Hold_Time,Total_Gap_Duration,Avg_Gap_per_call,Gap Duration After Leverage,Login Hours,Attendance
0,2026-03-30,Meta,Radhika,Deepa.negi@nexora.live,E-1443,Suja Basnet,TradeX,0,0,0,...,0,0,0,0,0,0,0.000000,0.0,0,0
1,2026-03-30,OJT,Saif,aamir@nexora.live,E-1911,Aamir,TradeX,247,221,26,...,3,3984,406,103,0,26606,107.716599,15491.0,09:29:50,Present
2,2026-03-30,OJT 1,Rehan,aaryan@nexora.live,E-2124,Aaryan,TradeX,200,186,33,...,12,6425,1898,1626,0,23452,117.260000,14452.0,09:17:57,Present
3,2026-03-30,1,Saif,abhinav.a@nexora.live,E-1356,Abhinav,TradeX,215,190,41,...,8,5994,1003,643,0,25839,120.181395,16164.0,09:50:33,Present
4,2026-03-30,Meta,Radhika,abhishek.k@nexora.live,E-1266,Abhishek Kumar,TradeX,356,229,55,...,33,14811,7256,6985,0,18942,53.207865,2922.0,10:22:33,Present
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,2026-03-30,Dialer,Rehan,vandana@nexora.live,E - 1677,Vandana,TradeX,0,0,0,...,0,0,0,0,0,0,0.000000,0.0,0,0
110,2026-03-30,Customer care,Vikas,vicky.v@nexora.live,E-1167,Vishal,TradeX,0,0,0,...,0,0,0,0,0,0,0.000000,0.0,0,0
111,2026-03-30,1,Saif,waman@nexora.live,E-1252,Waman Avhad,TradeX,222,170,95,...,51,13250,9051,8399,0,22305,100.472973,12315.0,10:52:35,Present
112,2026-03-30,OJT,Saif,yatin@nexora.live,E-1906,Yatin,TradeX,222,218,11,...,2,2217,135,78,0,31363,141.274775,21373.0,10:19:40,Present


In [104]:
formatted_df.isnull().sum()

Date                           0
Pool                           1
TL                             1
CRM ID                         0
Employee code                  0
Full Name                      0
Vertical                       0
Total_Dialed_Calls             0
Unique_Dialed_Numbers          0
Total_Connected_Calls          0
Total_Number_of_Call_Gap       0
Total_Call_GT_30               0
Total_Duration                 0
Total_Talk_Time                0
Total_Talk_Time_GT_30          0
Total_Connected_Hold_Time      0
Total_Gap_Duration             0
Avg_Gap_per_call               0
Gap Duration After Leverage    0
Login Hours                    0
Attendance                     0
dtype: int64

In [105]:
def seconds_to_hhmmss(seconds):
    if pd.isna(seconds):
        return None  # or "00:00:00" if you prefer
    total_seconds = round(seconds)
    hours = total_seconds // 3600
    remaining = total_seconds % 3600
    minutes = remaining // 60
    seconds = remaining % 60
    return f"{hours:02}:{minutes:02}:{seconds:02}"  # hh:mm:ss


columns_to_convert = [
    'Total_Duration', 
    'Total_Talk_Time', 
    'Total_Talk_Time_GT_30', 
    'Total_Connected_Hold_Time', 
    'Total_Gap_Duration',
    'Avg_Gap_per_call',
    'Gap Duration After Leverage'
]

for col in columns_to_convert:
    try:
        formatted_df[col] = formatted_df[col].apply(seconds_to_hhmmss)
    except Exception as e:
        print(f"Error in column: {col}")
        raise e  # re-raise the error so you still get the traceback


In [106]:
# from datetime import timedelta

# def seconds_to_hhmmss(seconds):
#     total_seconds = round(seconds)
#     hours = total_seconds // 3600
#     remaining = total_seconds % 3600
#     minutes = remaining // 60
#     seconds = remaining % 60
#     return f"{hours:02}:{minutes:02}:{seconds:02}"  # hh:mm:ss


# columns_to_convert = [
#     'Total_Duration', 
#     'Total_Talk_Time', 
#     'Total_Talk_Time_GT_30', 
#     'Total_Connected_Hold_Time', 
#     'Total_Gap_Duration',
#     'Avg_Gap_per_call',
#     'Gap Duration After Leverage'
# ]

# for col in columns_to_convert:
#     formatted_df[col] = formatted_df[col].apply(seconds_to_hhmmss)	


In [107]:
formatted_df = formatted_df.sort_values(by=['Date','CRM ID'])

In [108]:
formatted_df

,Date,Pool,TL,CRM ID,Employee code,Full Name,Vertical,Total_Dialed_Calls,Unique_Dialed_Numbers,Total_Connected_Calls,...,Total_Call_GT_30,Total_Duration,Total_Talk_Time,Total_Talk_Time_GT_30,Total_Connected_Hold_Time,Total_Gap_Duration,Avg_Gap_per_call,Gap Duration After Leverage,Login Hours,Attendance
0,2026-03-30,Meta,Radhika,Deepa.negi@nexora.live,E-1443,Suja Basnet,TradeX,0,0,0,...,0,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,0,0
1,2026-03-30,OJT,Saif,aamir@nexora.live,E-1911,Aamir,TradeX,247,221,26,...,3,01:06:24,00:06:46,00:01:43,00:00:00,07:23:26,00:01:48,04:18:11,09:29:50,Present
2,2026-03-30,OJT 1,Rehan,aaryan@nexora.live,E-2124,Aaryan,TradeX,200,186,33,...,12,01:47:05,00:31:38,00:27:06,00:00:00,06:30:52,00:01:57,04:00:52,09:17:57,Present
3,2026-03-30,1,Saif,abhinav.a@nexora.live,E-1356,Abhinav,TradeX,215,190,41,...,8,01:39:54,00:16:43,00:10:43,00:00:00,07:10:39,00:02:00,04:29:24,09:50:33,Present
4,2026-03-30,Meta,Radhika,abhishek.k@nexora.live,E-1266,Abhishek Kumar,TradeX,356,229,55,...,33,04:06:51,02:00:56,01:56:25,00:00:00,05:15:42,00:00:53,00:48:42,10:22:33,Present
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,2026-03-30,Dialer,Rehan,vandana@nexora.live,E - 1677,Vandana,TradeX,0,0,0,...,0,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,0,0
110,2026-03-30,Customer care,Vikas,vicky.v@nexora.live,E-1167,Vishal,TradeX,0,0,0,...,0,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,0,0
111,2026-03-30,1,Saif,waman@nexora.live,E-1252,Waman Avhad,TradeX,222,170,95,...,51,03:40:50,02:30:51,02:19:59,00:00:00,06:11:45,00:01:40,03:25:15,10:52:35,Present
112,2026-03-30,OJT,Saif,yatin@nexora.live,E-1906,Yatin,TradeX,222,218,11,...,2,00:36:57,00:02:15,00:01:18,00:00:00,08:42:43,00:02:21,05:56:13,10:19:40,Present


In [109]:
formatted_df['Attendance'].unique()

array([0, 'Present', 'Warning', 'Absent'], dtype=object)

In [110]:

formatted_df['Login Hours'] = formatted_df['Login Hours'].apply(lambda x: '00:00:00' if x == 0 else x)

formatted_df['Attendance'] = formatted_df['Attendance'].apply(lambda x: 'Absent' if x == 0 else x)


In [111]:
formatted_df

,Date,Pool,TL,CRM ID,Employee code,Full Name,Vertical,Total_Dialed_Calls,Unique_Dialed_Numbers,Total_Connected_Calls,...,Total_Call_GT_30,Total_Duration,Total_Talk_Time,Total_Talk_Time_GT_30,Total_Connected_Hold_Time,Total_Gap_Duration,Avg_Gap_per_call,Gap Duration After Leverage,Login Hours,Attendance
0,2026-03-30,Meta,Radhika,Deepa.negi@nexora.live,E-1443,Suja Basnet,TradeX,0,0,0,...,0,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,Absent
1,2026-03-30,OJT,Saif,aamir@nexora.live,E-1911,Aamir,TradeX,247,221,26,...,3,01:06:24,00:06:46,00:01:43,00:00:00,07:23:26,00:01:48,04:18:11,09:29:50,Present
2,2026-03-30,OJT 1,Rehan,aaryan@nexora.live,E-2124,Aaryan,TradeX,200,186,33,...,12,01:47:05,00:31:38,00:27:06,00:00:00,06:30:52,00:01:57,04:00:52,09:17:57,Present
3,2026-03-30,1,Saif,abhinav.a@nexora.live,E-1356,Abhinav,TradeX,215,190,41,...,8,01:39:54,00:16:43,00:10:43,00:00:00,07:10:39,00:02:00,04:29:24,09:50:33,Present
4,2026-03-30,Meta,Radhika,abhishek.k@nexora.live,E-1266,Abhishek Kumar,TradeX,356,229,55,...,33,04:06:51,02:00:56,01:56:25,00:00:00,05:15:42,00:00:53,00:48:42,10:22:33,Present
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,2026-03-30,Dialer,Rehan,vandana@nexora.live,E - 1677,Vandana,TradeX,0,0,0,...,0,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,Absent
110,2026-03-30,Customer care,Vikas,vicky.v@nexora.live,E-1167,Vishal,TradeX,0,0,0,...,0,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,00:00:00,Absent
111,2026-03-30,1,Saif,waman@nexora.live,E-1252,Waman Avhad,TradeX,222,170,95,...,51,03:40:50,02:30:51,02:19:59,00:00:00,06:11:45,00:01:40,03:25:15,10:52:35,Present
112,2026-03-30,OJT,Saif,yatin@nexora.live,E-1906,Yatin,TradeX,222,218,11,...,2,00:36:57,00:02:15,00:01:18,00:00:00,08:42:43,00:02:21,05:56:13,10:19:40,Present


In [112]:
# Convert 'Total Call Duration' to hh:mm:ss format
Dialers["Total Call Duration"] = Dialers["Total Call Duration"].apply(lambda x: str(x).split(" ")[-1])
Dialers['Number'] = "'" + Dialers['Number'].astype(str)


In [113]:
D = Dialers.drop(columns=['Talk Time (seconds)','Hold Time (seconds)','Total Duration (seconds)','Gap Duration (seconds)','Dialer'])
D['Number'] = D['Number'].astype(str).str.split('.').str[0]
D

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time,CRM ID,Employee code,Full Name,Pool,TL,Vertical,Call Gap,Gap Duration
0,Tata,2026-03-30,599 aamir,'917522088758,not connected,2026-03-30 10:20:18,00:00:01,00:00:00,00:00:00,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:00
1,Tata,2026-03-30,599 aamir,'917854014859,not connected,2026-03-30 10:59:03,00:00:31,00:00:00,00:00:00,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,Yes,00:38:44
2,Tata,2026-03-30,599 aamir,'919636559741,not connected,2026-03-30 11:01:20,00:00:31,00:00:00,00:00:00,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,Yes,00:01:46
3,Tata,2026-03-30,599 aamir,'919719218429,not connected,2026-03-30 11:06:48,00:00:31,00:00:00,00:00:00,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,Yes,00:04:57
4,Tata,2026-03-30,599 aamir,'919148696641,not connected,2026-03-30 11:07:29,00:00:04,00:00:00,00:00:00,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14474,Tata,2026-03-30,zaid,'918387958268,not connected,2026-03-30 18:17:30,00:00:31,00:00:00,00:00:00,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,Yes,00:01:11
14475,Tata,2026-03-30,zaid,'917385290403,not connected,2026-03-30 18:19:39,00:00:31,00:00:00,00:00:00,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,Yes,00:01:38
14476,Tata,2026-03-30,zaid,'919572640835,not connected,2026-03-30 18:20:23,00:00:31,00:00:00,00:00:00,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,No,00:00:13
14477,Tata,2026-03-30,zaid,'917008344146,not connected,2026-03-30 18:22:22,00:00:21,00:00:00,00:00:00,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,Yes,00:01:28


In [114]:
D.isnull().sum()

Source                 0
Date                   0
Dialer Name            0
Number                 0
Call Status            0
Call Start Time        0
Total Call Duration    0
Talk Time              0
Hold Time              0
CRM ID                 0
Employee code          0
Full Name              0
Pool                   0
TL                     0
Vertical               0
Call Gap               0
Gap Duration           0
dtype: int64

In [115]:
df = D.copy()

In [116]:
unique_calls = df.drop_duplicates(subset=["Date", "CRM ID", "Number","Call Status"])[["Date", "CRM ID", "Number","Call Status","Pool","TL","Full Name"]]

In [117]:
unique_calls

,Date,CRM ID,Number,Call Status,Pool,TL,Full Name
0,2026-03-30,aamir@nexora.live,'917522088758,not connected,OJT,Saif,Aamir
1,2026-03-30,aamir@nexora.live,'917854014859,not connected,OJT,Saif,Aamir
2,2026-03-30,aamir@nexora.live,'919636559741,not connected,OJT,Saif,Aamir
3,2026-03-30,aamir@nexora.live,'919719218429,not connected,OJT,Saif,Aamir
4,2026-03-30,aamir@nexora.live,'919148696641,not connected,OJT,Saif,Aamir
...,...,...,...,...,...,...,...
14474,2026-03-30,zaid.k@nexora.live,'918387958268,not connected,Pull Back,Parth,Zaid
14475,2026-03-30,zaid.k@nexora.live,'917385290403,not connected,Pull Back,Parth,Zaid
14476,2026-03-30,zaid.k@nexora.live,'919572640835,not connected,Pull Back,Parth,Zaid
14477,2026-03-30,zaid.k@nexora.live,'917008344146,not connected,Pull Back,Parth,Zaid


In [118]:
D.isnull().sum()

Source                 0
Date                   0
Dialer Name            0
Number                 0
Call Status            0
Call Start Time        0
Total Call Duration    0
Talk Time              0
Hold Time              0
CRM ID                 0
Employee code          0
Full Name              0
Pool                   0
TL                     0
Vertical               0
Call Gap               0
Gap Duration           0
dtype: int64

## Export


In [119]:
save_path = r'C:\Users\Akhil\Downloads\project\TradeX_report'
formatted_df.to_csv(f'{save_path}\\Summary_30_03.csv', index=False)
D.to_csv(f'{save_path}\\Dialer_30_03.csv', index=False)
crm_id_null_df.to_csv(f'{save_path}\\Not_Found_Users_30_03.csv', index=False)

In [120]:
df

,Source,Date,Dialer Name,Number,Call Status,Call Start Time,Total Call Duration,Talk Time,Hold Time,CRM ID,Employee code,Full Name,Pool,TL,Vertical,Call Gap,Gap Duration
0,Tata,2026-03-30,599 aamir,'917522088758,not connected,2026-03-30 10:20:18,00:00:01,00:00:00,00:00:00,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:00
1,Tata,2026-03-30,599 aamir,'917854014859,not connected,2026-03-30 10:59:03,00:00:31,00:00:00,00:00:00,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,Yes,00:38:44
2,Tata,2026-03-30,599 aamir,'919636559741,not connected,2026-03-30 11:01:20,00:00:31,00:00:00,00:00:00,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,Yes,00:01:46
3,Tata,2026-03-30,599 aamir,'919719218429,not connected,2026-03-30 11:06:48,00:00:31,00:00:00,00:00:00,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,Yes,00:04:57
4,Tata,2026-03-30,599 aamir,'919148696641,not connected,2026-03-30 11:07:29,00:00:04,00:00:00,00:00:00,aamir@nexora.live,E-1911,Aamir,OJT,Saif,TradeX,No,00:00:10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14474,Tata,2026-03-30,zaid,'918387958268,not connected,2026-03-30 18:17:30,00:00:31,00:00:00,00:00:00,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,Yes,00:01:11
14475,Tata,2026-03-30,zaid,'917385290403,not connected,2026-03-30 18:19:39,00:00:31,00:00:00,00:00:00,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,Yes,00:01:38
14476,Tata,2026-03-30,zaid,'919572640835,not connected,2026-03-30 18:20:23,00:00:31,00:00:00,00:00:00,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,No,00:00:13
14477,Tata,2026-03-30,zaid,'917008344146,not connected,2026-03-30 18:22:22,00:00:21,00:00:00,00:00:00,zaid.k@nexora.live,E-1275,Zaid,Pull Back,Parth,TradeX,Yes,00:01:28
